## Metadata Extraction

07/10/2026

In [1]:
import pandas as pd
dir_path = "/Users/ameliecoy/Library/CloudStorage/OneDrive-SharedLibraries-TheUniversityofManchester/Cheuk ning sharon Chau - Sputum Photos/Sputum Photos RG"
#dir_path = '/Users/ameliecoy/Downloads/Sputum Photos'
#dir_path = "/Users/ameliecoy/Library/CloudStorage/OneDrive-SharedLibraries-TheUniversityofManchester/Cheuk ning sharon Chau - Sputum Photos"



In [ ]:
from PIL import Image
from PIL.ExifTags import TAGS
import os
import re


# Get all files in dir_path
files = os.listdir(dir_path)


# Get patient ID from filename if present
def get_patient_num(filename):
    if filename.lower().startswith('pt'):
        pt = filename.split(' ')[0][2:]  # Extract the number after 'pt'
        return pt

    return None

# Get lighting location from filename if present
def get_lighting_location(filename):
    # find 'location*' or 'location *' in filename
    location_match = re.search(r'location\s*(\d+)', filename, re.IGNORECASE)
    if location_match:
        return int(location_match.group(1))

    # account for typo - 'lcoation'
    location_match = re.search(r'lcoation\s*(\d+)', filename, re.IGNORECASE)
    if location_match:
        return int(location_match.group(1))

    return 1 # If no location recorded, assume location 1 (as per Rachel Gyte)

# Extract metadata from image file
def extract_metadata(filename, site, photographer, lighting_location):
    metadata_dict = {}
    # Define desired keys
    keys = ['filename', 'site', 'photographer', 'DateTime', 'patient_id', 'lighting_location', 'Model', 'YCbCrPositioning', 'XResolution', 'YResolution']

    # Add parameter metadata
    metadata_dict['filename'] = filename
    metadata_dict['site'] = site
    metadata_dict['photographer'] = photographer
    metadata_dict['lighting_location'] = lighting_location
    metadata_dict['patient_id'] = get_patient_num(filename)

    img = Image.open(f"{dir_path}/{filename}")
    exifdata = img.getexif()

    # Looping through all the tags present in exifdata
    for tagid in exifdata:

        tagname = TAGS.get(tagid, tagid)
        value = exifdata.get(tagid)
        if tagname in keys:
            metadata_dict[tagname] = value


    # Rename DateTime to date, Model to phone_model
    if 'DateTime' in metadata_dict:
        metadata_dict['date'] = metadata_dict.pop('DateTime')
        # Convert to datetime object
        metadata_dict['date'] = pd.to_datetime(metadata_dict['date'], format='%Y:%m:%d %H:%M:%S', errors='coerce')
    if 'Model' in metadata_dict:
        metadata_dict['phone_model'] = metadata_dict.pop('Model')

    # Fill any empty dates
    if 'date' not in metadata_dict or pd.isnull(metadata_dict['date']):
        metadata_dict['date'] = None

    return metadata_dict


# Open the images and send to metadata extraction functions
all_files_metadata = []
for file in files:
    fsize = os.path.getsize(f"{dir_path}/{file}")
    if fsize > 0 :
        try:
            img = Image.open(f"{dir_path}/{file}")
        except Exception as e:
            print(f"Error opening image {file}: {e}")
        lighting_location = get_lighting_location(file)
        metadata_dict = extract_metadata(file, site = 'MCR Clinic', photographer = 'Rachel Gyte', lighting_location = lighting_location)
        all_files_metadata.append(metadata_dict)


# Remove NoneType dates temporarily to avoid errors when sorting
empty_dates = [metadata for metadata in all_files_metadata if metadata.get('date') is None]
orderable_dates = all_files_metadata.copy()
for metadata in empty_dates:
    orderable_dates.remove(metadata)

# Order by date and print filename and date
orderable_dates.sort(key=lambda x: x.get('date', None))
all_files_metadata = orderable_dates + empty_dates  # Combine the ordered and unordered lists
for metadata in all_files_metadata:
    print(f"Filename: {metadata['filename']}, Date: {metadata['date']}, Patient ID: {metadata['patient_id']}, Lighting Location: {metadata['lighting_location']}")



Filename: Photo 15-04-2026, 12 41 19.jpg, Date: 2026-04-15 12:41:19, Patient ID: None, Lighting Location: 1
Filename: Photo 15-04-2026, 12 41 27.jpg, Date: 2026-04-15 12:41:27, Patient ID: None, Lighting Location: 1
Filename: Photo 15-04-2026, 12 42 05.jpg, Date: 2026-04-15 12:42:05, Patient ID: None, Lighting Location: 1
Filename: Photo 21-04-2026, 12 57 41.jpg, Date: 2026-04-21 12:57:41, Patient ID: None, Lighting Location: 1
Filename: Photo 21-04-2026, 12 57 45.jpg, Date: 2026-04-21 12:57:45, Patient ID: None, Lighting Location: 1
Filename: Photo 21-04-2026, 12 58 02.jpg, Date: 2026-04-21 12:58:02, Patient ID: None, Lighting Location: 1
Filename: Photo 21-04-2026, 12 58 14.jpg, Date: 2026-04-21 12:58:14, Patient ID: None, Lighting Location: 1
Filename: Photo 23-04-2026, 16 53 27.jpg, Date: 2026-04-23 16:53:27, Patient ID: None, Lighting Location: 1
Filename: Photo 23-04-2026, 16 54 26.jpg, Date: 2026-04-23 16:54:26, Patient ID: None, Lighting Location: 1
Filename: Photo 23-04-2026, 

In [ ]:
# order by date, check if the unnamed photos are all recent ones or not - if so then tell rachel i need the patient ID
# OCR or filename maybe used for patient id
# Learn about colour spaces